# 从 Whisper 迁移到 GPT-Transcribe 和 GPT-Live-Transcribe

## 概述

OpenAI 新推出的转录模型为现有 Whisper 用户提供了两条明确的升级路径。GPT-Transcribe（`gpt-transcribe`）专为通过上传文件对已完成的音频进行准确转录而设计。GPT-Live-Transcribe（`gpt-live-transcribe`）专为低延迟、持续流式的转录而设计。

总体而言，API 端点保持不变。文件请求仍然使用 POST /v1/audio/transcriptions。Realtime 集成仍然是配置转录会话并处理转录事件。不同之处在于支持的请求字段、输出格式，以及上下文/多语言提示。

### 保持不变的部分

- 文件转录使用现有的 Audio API 转录端点。
- 实时转录使用现有的 Realtime 转录会话、WebSocket 或 WebRTC 传输，以及转录事件。
- 应用仍然提供音频、接收转录文本，并处理 API 或连接错误。

### 发生变化的部分

- 对于已完成的文件、流式文件转录或已提交的 Realtime 轮次，使用 `gpt-transcribe`。
- 对于持续流式、低延迟的实时转录，使用 `gpt-live-transcribe`。
- 将旧版 `language` 提示替换为新模型的 `languages` 数组。
- 使用 `prompt` 提供自由格式的上下文，使用 `keywords` 提供预期会在音频中出现的字面词条。
- 处理 GPT-Transcribe 的检测语言输出，包括空的 languages 数组。

## 1. 选择迁移路径

- 录制的会议、通话或上传的音频：在 POST /v1/audio/transcriptions 上将 `whisper-1` 迁移到 `gpt-transcribe`。
- 实时字幕或持续到达的麦克风音频：在 Realtime 转录会话中将 `gpt-realtime-whisper` 迁移到 `gpt-live-transcribe`。
- 在手动提交音频轮次后进行更高准确度的转录：通过 WebSocket 在 Realtime 转录会话中使用 `gpt-transcribe`。
请将实时音频和流式输出视为两个独立的决策。已完成的文件可以产生流式转录事件，而 Realtime 会话也可以等待音频轮次提交后再产生增量。

## 2. 迁移录制音频

最小的文件迁移会保留端点、SDK 调用和文件上传。替换模型标识符，然后处理 JSON 格式的转录响应。


### 前提条件

- Python 3.11 或更高版本。
- OpenAI Python SDK：`python -m pip install --upgrade openai`。
- 一个 `OPENAI_API_KEY` 环境变量。
- 一个名为 `meeting.wav` 的完整录音，或修改 `audio_path` 指向你自己的文件。
- 能够访问示例中使用的转录模型。

文件示例会发起真实的 API 请求。Realtime 示例会构造会话负载。对于 `gpt-live-transcribe`，使用 WebSocket 或 WebRTC；对于提交轮次模式的 `gpt-transcribe`，使用 WebSocket。


In [ ]:
import json
from pathlib import Path

from openai import OpenAI

client = OpenAI()
audio_path = Path("meeting.wav")


**迁移前：Whisper 文件转录**


In [ ]:
with audio_path.open("rb") as audio:
    before = client.audio.transcriptions.create(
        model="whisper-1",
        file=audio,
    )

print(before.text)


**迁移后：GPT-Transcribe**


In [ ]:
with audio_path.open("rb") as audio:
    after = client.audio.transcriptions.create(
        model="gpt-transcribe",
        file=audio,
        response_format="json",
    )

print(after.text)


**注意事项**

- 请注意，新模型返回 JSON。如果现有应用请求 `text`、`verbose_json`、`srt` 或 `vtt`，不要假定相同的 response_format 仍然有效。对于这些格式，请继续使用 Whisper，或者显式构建并测试应用所需的下游转换。

## 3. 更新语言提示和领域上下文

旧版 Whisper 集成通常只发送单个 `language` 值，并将领域词汇放在自由格式的 prompt 中。新模型可以接受多种预期的输入语言，并将结构化的关键词提示与通用 prompt 分开。


**迁移前：单语言 Whisper 请求**


In [ ]:
from openai import OpenAI

client = OpenAI()

with open("meeting.wav", "rb") as audio:
    result = client.audio.transcriptions.create(
        model="whisper-1",
        file=audio,
        language="en",
        prompt="A support call about AC-42.",
    )

print(result.text)


**迁移后：多语言 GPT-Transcribe 请求**


In [ ]:
from openai import OpenAI

client = OpenAI()

with open("meeting.wav", "rb") as audio:
    result = client.audio.transcriptions.create(
        model="gpt-transcribe",
        file=audio,
        prompt="A customer support call about billing.",
        extra_body={
            "keywords": ["AC-42", "Premium Plus"],
            "languages": ["en", "fr"],
        },
    )

print(result.text)


**注意事项**

- 使用 `prompt` 描述录音的主题或场景。
- 使用 `keywords` 提供可能真正被说出的字面产品名称、药品名称、账户标识符或其他领域特定术语。
- 使用 `languages` 指明预期的口语语言。
- 不要同时发送 `language` 和 `languages`。
- 在发出请求前验证关键词输入：每个关键词都必须是单行字面值，且不能包含 `<`、`>`、回车符或换行符。无效的关键词输入会导致整个请求或会话更新被拒绝。

## 4. 流式转录已完成的文件

流式处理针对的是转录输出，而不是音频输入。应用首先提供一个已完成的录音，然后 API 在转录该录音时流式返回文本。


**迁移前：阻塞式 Whisper 文件请求**


In [ ]:
with audio_path.open("rb") as audio:
    before = client.audio.transcriptions.create(
        model="whisper-1",
        file=audio,
    )

print(before.text)

# whisper-1 does not provide transcript.text.delta events.


**迁移后：流式 GPT-Transcribe 文件请求**


In [ ]:
with audio_path.open("rb") as audio:
    stream = client.audio.transcriptions.create(
        model="gpt-transcribe",
        file=audio,
        stream=True,
    )

    for event in stream:
        if event.type == "transcript.text.delta":
            print(event.delta, end="", flush=True)
        elif event.type == "transcript.text.done":
            print("\nFinal:", event.text)


**注意事项**

- 使用转录事件区分进行中的转录和已完成的转录。用 `transcript.text.delta` 事件更新转录内容或进度 UI，然后用 `transcript.text.done` 将转录标记为最终结果。
- 不要把文件流式处理称为实时音频采集：文件必须已经存在。

## 5. 迁移持续实时转录

对于实时字幕工作流，请保留 Realtime 连接和转录会话架构。更换转录模型并更新模型特定的配置。


**迁移前：GPT-Realtime-Whisper**


In [ ]:
legacy_realtime_session = json.loads(
    r'''
{
  "type": "session.update",
  "session": {
    "type": "transcription",
    "audio": {
      "input": {
        "transcription": {
          "model": "gpt-realtime-whisper",
          "language": "en"
        },
        "turn_detection": null
      }
    }
  }
}
'''
)

print(json.dumps(legacy_realtime_session, indent=2))


**迁移后：GPT-Live-Transcribe**


In [ ]:
live_transcription_session = json.loads(
    r'''
{
  "type": "session.update",
  "session": {
    "type": "transcription",
    "audio": {
      "input": {
        "transcription": {
          "model": "gpt-live-transcribe",
          "prompt": "A support call about a plan.",
          "keywords": ["AC-42", "Premium Plus"],
          "languages": ["en", "fr"],
          "delay": "low"
        },
        "turn_detection": null
      }
    }
  }
}
'''
)

print(json.dumps(live_transcription_session, indent=2))


**注意事项**

- 继续处理 `conversation.item.input_audio_transcription.delta` 和 `conversation.item.input_audio_transcription.completed`。
- 保留现有基于 item_id 的对账逻辑。
- 使用模型的 `delay` 设置，在更快的部分转录与更准确的最终转录之间进行权衡。
- 保留现有的 WebSocket 或 WebRTC 传输；Realtime 转录两者都支持。

## 6. 在提交轮次模式的 Realtime 会话中使用 GPT-Transcribe

GPT-Transcribe 并不仅限于上传的文件。当你的应用需要在手动提交音频轮次后进行更高准确度的转录时，可以通过 WebSocket 在 Realtime 会话中将其用作输入转录模型。


**迁移前：旧版实时转录会话**


In [ ]:
legacy_live_session = json.loads(
    r'''
{
  "type": "session.update",
  "session": {
    "type": "transcription",
    "audio": {
      "input": {
        "transcription": {
          "model": "gpt-realtime-whisper",
          "language": "en"
        }
      }
    }
  }
}
'''
)

print(json.dumps(legacy_live_session, indent=2))


**迁移后：提交轮次模式的 GPT-Transcribe 会话**


In [ ]:
committed_turn_session = json.loads(
    r'''
{
  "type": "session.update",
  "session": {
    "type": "transcription",
    "audio": {
      "input": {
        "transcription": {
          "model": "gpt-transcribe",
          "prompt": "A customer support call.",
          "keywords": ["AC-42"],
          "languages": ["en", "fr"]
        },
        "turn_detection": null
      }
    }
  }
}
'''
)

print(json.dumps(committed_turn_session, indent=2))


**注意事项**

- 将音频追加到输入缓冲区，然后发送 `input_audio_buffer.commit`。转录按每个已提交的音频块进行，而不是持续的实时字幕。GPT-Transcribe 可能会在该音频块的完成事件之前发出转录增量事件。

## 7. 更新响应和事件处理

GPT-Transcribe 可以在已完成的文件结果和 Realtime 结果中包含检测到的输入语言。这些是模型返回的预测，而不是对调用方语言提示的复述。


**迁移前：不含语言预测的转录结果**


In [ ]:
legacy_transcription_response = json.loads(
    r'''
{
  "text": "Bonjour, pouvez-vous m'entendre ?"
}
'''
)

print(json.dumps(legacy_transcription_response, indent=2))


**迁移后：GPT-Transcribe 检测语言响应**


In [ ]:
detected_language_response = json.loads(
    r'''
{
  "text": "Bonjour, pouvez-vous m'entendre ?",
  "languages": [
    { "code": "fr" }
  ]
}
'''
)

print(json.dumps(detected_language_response, indent=2))

language_codes = [
    item["code"]
    for item in detected_language_response.get("languages", [])
]
print("Detected languages:", language_codes)


**注意事项**

- 当模型无法做出可靠预测时，将 `languages: []` 作为有效结果处理。
- 不要据此推断没有人说话、请求失败或返回了一个合成的 `unknown` 语言。GPT-Live-Transcribe 在其初始约定中不返回检测语言预测。
- 对于 Realtime，请检查 `conversation.item.input_audio_transcription.completed`；GPT-Transcribe 可能会在该事件上附加相同的语言元数据。

## 8. 切换前检查兼容性

一些现有的转录功能在 `gpt-transcribe` 或 `gpt-live-transcribe` 中没有可直接替换的方案。

- 字幕：如果集成依赖原生 SRT 或 VTT 输出，请保留 `whisper-1`。
- 时间戳：保留明确支持单词或片段时间戳的模型和响应格式。
- 翻译：如果应用需要将音频翻译成英文，请保留 `/v1/audio/translations` 并使用受支持的翻译模型。
- 说话人分离：当应用需要说话人标签时，在 `/v1/audio/transcriptions` 上使用 `gpt-4o-transcribe-diarize`，并设置 `response_format: "diarized_json"`。对于超过 30 秒的录音，请设置 `chunking_strategy: "auto"`。该模型不支持 Realtime API。

## 9. 迁移前后进行评估

在两个模型上使用相同的代表性音频片段。比较基线、仅替换模型的直接替换版本，以及添加了新上下文字段的版本。这样可以区分模型本身带来的改进与 prompt、keywords 或语言提示带来的改进。

### 评估转录质量

- 在代表性录音上的整体词错误率。
- 人名、产品名称、药品名称、账户标识符、数字和电子邮件地址的精确匹配准确率。
- 在口音、背景噪音、电话音频和简短话语下的表现。
- 语言切换和混合语言语音中的准确率。
- 关键词幻觉：确认转录结果没有插入实际上并未说出的提示词。
- 最终转录结果以及所有预期语言元数据的完整性。

### 评估流式和运行行为

- 首个转录增量的到达时间，分别报告文件流式处理和实时音频的情况。
- 从显式提交轮次到最终转录结果的时间。
- 中位数延迟和尾部延迟，包括 p95 或团队现有的服务水平阈值。
- 部分转录文本在完成前变化的频率。
- 重叠轮次之间的事件顺序和 item_id 对账。
- 重连、重试、空音频和会话中断时的行为。
- 以正式发布的权威来源为准的实际请求定价和生产环境速率限制。

## 常见迁移错误

- 仅因为 `gpt-transcribe` 针对已完成音频进行了优化，就认为它与 Realtime 不兼容。
- 因为处理在提交后才开始，就认为 GPT-Transcribe 无法流式输出。
- 以为流式文件输出意味着该端点可以接收持续的麦克风流。
- 同时发送旧版 `language` 和新模型的 `languages`。
- 期望不支持相应功能的模型输出字幕格式、时间戳、说话人标签或检测语言。
- 将空的检测语言数组视为 API 错误。
- 强行把关键词写入转录结果，而不是将其视为提示。
- 以为所有 Realtime 模型都支持相同的语音活动检测和延迟控制。


## 相关资源

- [语音转文本](https://developers.openai.com/api/docs/guides/speech-to-text)
- [实时转录](https://developers.openai.com/api/docs/guides/realtime-transcription)
- [GPT-4o Transcribe Diarize](https://developers.openai.com/api/docs/models/gpt-4o-transcribe-diarize)